In [6]:
import polars as pl
pl.Config.set_tbl_rows(15)

polars.config.Config

In [7]:
x = pl.read_parquet(r"C:\Users\abrosnan\Documents\PhD_1\Experiments\Dominance hierachy\Alpha LSD\LSD 2\LSD_POK_3\LSD_DOMINANT\codex_2026-06-21\results\main_df.parquet")

In [8]:
x

antenna,time_under,animal_id,datetime,phase,day,hour,phase_count,time_spent,position
u8,duration[μs],enum,"datetime[μs, Europe/Warsaw]",enum,u16,u8,u16,f64,cat
8,256ms,"""2BFEF91A04""",2025-08-06 11:57:10.647 CEST,"""light_phase""",1,11,1,0.0,"""undefined"""
7,996ms,"""2BFEF91A04""",2025-08-06 11:57:12.230 CEST,"""light_phase""",1,11,1,1.58,"""c1_c4"""
1,534ms,"""EC4BFD1A04""",2025-08-06 11:57:12.237 CEST,"""light_phase""",1,11,1,0.0,"""undefined"""
5,205ms,"""ACB7F91A04""",2025-08-06 11:57:12.720 CEST,"""light_phase""",1,11,1,0.0,"""undefined"""
6,359ms,"""ACB7F91A04""",2025-08-06 11:57:13.867 CEST,"""light_phase""",1,11,1,1.15,"""c3_c4"""
6,319ms,"""33F3F91A04""",2025-08-06 11:57:15.087 CEST,"""light_phase""",1,11,1,0.0,"""undefined"""
5,320ms,"""33F3F91A04""",2025-08-06 11:57:16.014 CEST,"""light_phase""",1,11,1,0.93,"""c4_c3"""
2,1s 723ms,"""EC4BFD1A04""",2025-08-06 11:57:16.100 CEST,"""light_phase""",1,11,1,3.86,"""c1_c2"""
…,…,…,…,…,…,…,…,…,…


In [9]:
import polars as pl

# Load parquet
df = x

# Distance between antennas on each tube
TUBE_DISTANCE_CM = 20

# Tube/corridor positions only (check to see if they are faster towards or away from food/water)
tube_positions = [
    "c1_c2", "c2_c1",
    "c2_c3", "c3_c2",
    "c3_c4", "c4_c3",
    "c4_c1", "c1_c4",
]

# Keep only tube-crossing rows and calculate speed
tube_df = (
    df
    .filter(pl.col("position").is_in(tube_positions))
    .with_columns(
        (TUBE_DISTANCE_CM / pl.col("time_spent")).alias("speed_cm_s"),
        (TUBE_DISTANCE_CM / pl.col("time_spent") / 100).alias("speed_m_s"),
    )
)

# Optional filter: remove implausibly long tube detections
# This keeps crossings where time_spent <= 10 seconds
filtered = tube_df.filter(pl.col("time_spent") <= 10)

# Count excluded events per day and animal
excluded_counts = (
    tube_df
    .filter(pl.col("time_spent") > 10)
    .group_by("day", "animal_id")
    .agg(
        pl.len().alias("excluded_over_10s")
    )
)

# Summarise average speed per day and animal
summary = (
    filtered
    .group_by("day", "animal_id")
    .agg(
        pl.len().alias("crossings_used"),
        pl.mean("speed_cm_s").alias("mean_speed_cm_s"),
        pl.median("speed_cm_s").alias("median_speed_cm_s"),
        pl.min("speed_cm_s").alias("min_speed_cm_s"),
        pl.max("speed_cm_s").alias("max_speed_cm_s"),
    )
    .join(excluded_counts, on=["day", "animal_id"], how="left")
    .with_columns(
        pl.col("excluded_over_10s").fill_null(0).cast(pl.Int64),
        pl.col("mean_speed_cm_s").round(3),
        pl.col("median_speed_cm_s").round(3),
        pl.col("min_speed_cm_s").round(3),
        pl.col("max_speed_cm_s").round(3),
    )
    .select(
        "day",
        "animal_id",
        "crossings_used",
        "excluded_over_10s",
        "mean_speed_cm_s",
        "median_speed_cm_s",
        "min_speed_cm_s",
        "max_speed_cm_s",
    )
    .sort("day", "animal_id")
)


In [10]:
summary.sort(by=["day", "mean_speed_cm_s"], descending=[False, True])

animal_id,crossings_used,excluded_over_10s,mean_speed_cm_s,median_speed_cm_s,min_speed_cm_s,max_speed_cm_s
enum,u32,i64,f64,f64,f64,f64
"""5B39FA1A04""",20635,265,37.885,37.037,2.022,105.263
"""7E0CFA1A04""",15688,138,35.027,33.333,2.035,111.111
"""33F3F91A04""",12487,107,33.863,33.333,2.008,105.263
"""C212FA1A04""",12819,78,33.128,31.746,2.01,105.263
"""EC4BFD1A04""",16071,97,31.435,30.769,2.062,111.111
"""2BFEF91A04""",14406,54,29.629,28.986,2.094,95.238
"""2BD2F91A04""",12866,108,29.521,28.571,2.0,105.263
"""6AA1F91A04""",18319,116,29.263,27.778,2.026,111.111
"""ACB7F91A04""",13149,142,29.006,26.667,2.062,111.111
